# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step exploration of the FAIR² dataset on adoption predictors in rangeland management, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant JSON-LD)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset and its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

# Print further description and high-level statistics
print("\n---\nData collection:")
print(metadata.dataCollection)
print("\nData timeframe:")
print(metadata.dataCollectionTimeframe)
print("\nFields with personal or sensitive information:")
print(metadata.personalSensitiveInformation)


## 2. Data Overview
Review available record sets, fields, and their `@id`.

> **Note**: Each entity (record set, field, etc.) is referenced by its `@id` for full traceability per Croissant best practices.

In [ ]:
# Print the available record sets and their fields by their @id.

record_set_ids = []
for rset in dataset.record_sets:
    record_set_ids.append(rset['@id'])
    print(f"Record Set @id: {rset['@id']} | Name: {rset.get('name','')} ")
    if 'field' in rset:
        print('  Fields:')
        for f in rset['field']:
            if isinstance(f, dict):
                print(f"    - Field @id: {f.get('@id','')} | Name: {f.get('name','')}")
            else:
                print(f"    - Field @id: {f}")
    print() 

if not record_set_ids:
    print('No explicit record sets found in the metadata. Attempting to infer available record sets.')
    for rset in dataset._manifest.get('recordSet', []):
        print(f"Possibly available Record Set @id: {rset.get('@id','')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

*Use the exact `@id` for each available record set and field from the overview section above. Example below uses all found record set @ids.*

In [ ]:
# If record sets were identified above, use their @ids for extraction.
dataframes = {}
if not record_set_ids:
    # Try to find the record sets from the manifest if missing
    record_set_ids = [r.get('@id') for r in dataset._manifest.get('recordSet', [])]
    if not record_set_ids:
        raise RuntimeError("No record sets available to load.")

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"No records found for Record Set @id: {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for Record Set @id: {record_set_id} | shape: {df.shape}")
    print(f"Columns (raw field @ids): {df.columns.tolist()}")
    print(df.head(2))

# For demonstration, select the first available record set for deep EDA
selected_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common EDA steps such as filtering, normalizing numeric fields, and grouping/categorizing. 

*Below EDA is applied for the first extracted record set and its fields, referencing them by `@id`*. 

> Make sure to replace `numeric_field_id` and `group_field_id` with valid field @ids from above. This example assumes common numeric and grouping fields exist.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Inspect possible numeric fields in the selected record set
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"Fields in record set @{selected_record_set_id}: {df.columns.tolist()}")
    # Try to auto-select a numeric field (e.g., coefficient or p-value)
    numeric_field_id = None
    for col in df.columns:
        try:
            arr = pd.to_numeric(df[col], errors='coerce')
            if arr.notnull().sum() > 0 and arr.dtype in [np.float64, np.int64, float, int]:
                numeric_field_id = col
                break
        except:
            continue
    if not numeric_field_id:
        print('No numeric field automatically detected. Please inspect your data and set numeric_field_id manually.')
    else:
        print(f"Auto-selected numeric field for EDA: {numeric_field_id}")

    threshold = np.nanmedian(pd.to_numeric(df[numeric_field_id], errors='coerce'))
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.3g}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') -
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to select a grouping/class field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() > 1 and df[col].dtype == 'object':
            group_field_id = col
            print(f"Using group-by field: {group_field_id}")
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields using matplotlib and seaborn. Shows numeric field distribution and (if available) groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    plt.figure(figsize=(6,4))
    values = pd.to_numeric(df[numeric_field_id], errors='coerce')
    sns.histplot(values.dropna(), kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,3))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- This notebook demonstrated how to access Croissant-based datasets via `mlcroissant`, inspect schema elements by `@id`, and extract records for analysis.
- Key numeric fields can be filtered and normalized for further machine learning or statistical analyses.
- Use the record set and field `@id`s in all references for code clarity and reproducibility.

> See the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/) for more advanced features—including joining record sets, column-level extraction, and metadata enrichment.